# 5. Trade-off, rewards and multi-step generation

The fine-tuning of notebook 4 is repeated with different weights $\beta$ of the KL penalty and with different
rewards, and the generated lists are examined position by position.

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import pandas as pd

from gptrec.data.batches import build_eval_examples
from gptrec.data.log import InteractionLog
from gptrec.evaluation.policy_report import EvaluationConfig, evaluate_policy, metrics_table, per_step_table
from gptrec.paths import ARTIFACTS_DIR, PROCESSED_DATA_DIR
from gptrec.rollout import GenerationConfig, RatioBaseline, RewardSpec, SimulatedFeedback
from gptrec.training.checkpoint import load_policy, load_reward_model, save_model
from gptrec.training.precision import default_device
from gptrec.training.rl import RLConfig, finetune_with_rl

device = default_device()
log = InteractionLog.load(PROCESSED_DATA_DIR / "yambda-50m")

In [ ]:
pretrained = load_policy(ARTIFACTS_DIR / "policy_pretrained.pt", device)
reward_models = {
    "train": load_reward_model(ARTIFACTS_DIR / "reward_model_a.pt", device),
    "heldout": load_reward_model(ARTIFACTS_DIR / "reward_model_b.pt", device),
}

HORIZON = 10
HISTORY_LEN = pretrained.config.transformer.max_positions - HORIZON
MAX_FUTURE = 20
evaluation_config = EvaluationConfig(ks=(1, 5, 10), generation=GenerationConfig(horizon=HORIZON, greedy=True))
test_examples = build_eval_examples(log, "test", HISTORY_LEN, MAX_FUTURE)

base_config = RLConfig(num_steps=1000, history_len=HISTORY_LEN, max_future=MAX_FUTURE, generation=GenerationConfig(horizon=HORIZON))


def finetuned(run_name: str, config: RLConfig):
    path = ARTIFACTS_DIR / "sweep" / f"{run_name}.pt"
    if not path.exists():
        policy, _ = finetune_with_rl(pretrained, reward_models["train"], log, config, device)
        save_model(policy, path)
    return load_policy(path, device)


def evaluate(policy, config: EvaluationConfig = evaluation_config):
    return evaluate_policy(policy, test_examples, log.items, reward_models, config, device)


SUMMARY_ROWS = [
    "top_k/all/ndcg@10",
    "next_k/all/ndcg@10",
    "next_k/liked/ndcg@10",
    "next_k/skipped/recall@10",
    "predicted/train/played_ratio",
    "predicted/heldout/played_ratio",
    "predicted/heldout/personal_played_ratio",
    "predicted/heldout/played_seconds",
    "predicted/heldout/session_continues",
    "predicted/heldout/returns",
    "next_k/profile/catalog_coverage",
    "next_k/profile/mean_user_pair_overlap",
    "next_k/profile/head_item_share",
    "next_k/profile/mean_duration_seconds",
    "next_k/profile/history_repeat_rate",
]

## Relevance against reward

The fine-tuning is repeated for several values of $\beta$. Each run is a point: the predicted played ratio of its
lists against their NDCG@K. The ratio is predicted by $R_{\text{train}}$ and by $R_{\text{held-out}}$; a gap between
the two is a gain that only the optimised model predicts.

In [ ]:
kl_weights = [0.0, 0.1, 0.2, 0.3, 0.5, 1.0]
kl_sweep = {"pretrained": evaluate(pretrained)}
for kl_weight in kl_weights:
    kl_sweep[f"kl={kl_weight}"] = evaluate(finetuned(f"kl_{kl_weight}", replace(base_config, reference_kl_weight=kl_weight)))

kl_table = metrics_table(kl_sweep)
kl_table.loc[SUMMARY_ROWS]

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for axis, relevance in zip(axes, ["top_k/all/ndcg@10", "next_k/liked/ndcg@10"]):
    for judge in reward_models:
        axis.plot(kl_table.loc[f"predicted/{judge}/played_ratio"], kl_table.loc[relevance], marker="o", label=f"reward model: {judge}")
    for run_name in kl_table.columns:
        axis.annotate(run_name, (kl_table.loc["predicted/heldout/played_ratio", run_name], kl_table.loc[relevance, run_name]), fontsize=8)
    axis.set(xlabel="predicted played ratio of the generated list", ylabel="NDCG@10", title=relevance)
    axis.legend()
figure.tight_layout()

## Rewards

The reward generalises to

$$r_k = w_\rho\,\big(\hat\rho_k - b_k\big) + w_c\, \hat c_k + w_u\, \hat u_k + w_F\, \frac{\mathbb{1}[a_k \in F]}{\log_2(k + 1)},$$

where $F$ is the set of the next $M$ items the user consumed after $h$, and $b_k$ is a baseline.

| Reward | $w_\rho$ | $b_k$ | $w_c$ | $w_u$ | $w_F$ |
|---|---|---|---|---|---|
| played ratio | 1 | 0 | 0 | 0 | 0 |
| session | 0 | | 1 | 0 | 0 |
| return | 0 | | 0 | 1 | 0 |
| ratio + return | 1 | 0 | 0 | 1 | 0 |
| personal ratio | 1 | $\hat\rho(a_k \mid h', a_{1:k-1})$ | 0 | 0 | 0 |
| ratio above item mean | 1 | $\bar\rho(a_k)$ | 0 | 0 | 0 |
| ratio + logged relevance | 1 | 0 | 0 | 0 | 10 |

Here $h'$ is the history of another user of the batch and $\bar\rho(j)$ the mean played ratio of item $j$ in the
training period. For a policy that ignores the user, the personal ratio has zero expectation. *Played time*
replaces $\hat\rho_k$ by $\hat\rho_k\, d(a_k) / \operatorname{median}(d)$, where $d$ is the duration of an item. All
runs use the same $\beta$.

In [ ]:
rewards = {
    "played ratio": RewardSpec(),
    "played time": RewardSpec(weigh_ratio_by_duration=True),
    "session": RewardSpec(played_ratio_weight=0.0, session_continues_weight=1.0),
    "return": RewardSpec(played_ratio_weight=0.0, returns_weight=1.0),
    "ratio + return": RewardSpec(returns_weight=1.0),
    "personal ratio": RewardSpec(ratio_baseline=RatioBaseline.OTHER_USER),
    "ratio above item mean": RewardSpec(ratio_baseline=RatioBaseline.ITEM_MEAN),
    "ratio + logged relevance": RewardSpec(logged_relevance_weight=10.0),
}
reward_sweep = {"pretrained": kl_sweep["pretrained"]}
for reward_name, reward in rewards.items():
    reward_sweep[reward_name] = evaluate(finetuned(f"reward_{reward_name.replace(' ', '_')}", replace(base_config, reward=reward)))

metrics_table(reward_sweep).loc[SUMMARY_ROWS]

## Multi-step generation

For each list position $k$: the probability that $a_k \in F$, the entropy of $\pi_\theta(\cdot \mid s_k)$, the
predicted played ratio, and the disagreement $|\hat\rho^{\,\text{train}}_k - \hat\rho^{\,\text{held-out}}_k|$ between the
two reward models.

In [ ]:
policies = {"pretrained": pretrained, "rl": load_policy(ARTIFACTS_DIR / "policy_rl.pt", device)}
step_tables = {name: per_step_table(evaluate(policy), test_examples, log.items) for name, policy in policies.items()}

columns = ["in_true_continuation", "policy_entropy", "predicted_played_ratio/train", "played_ratio_disagreement/train_vs_heldout"]
figure, axes = plt.subplots(1, len(columns), figsize=(5 * len(columns), 3.5))
for axis, column in zip(axes, columns):
    pd.DataFrame({name: table[column] for name, table in step_tables.items()}).plot(ax=axis, marker="o", title=column)
figure.tight_layout()

### Simulated feedback

A generated item has no observed feedback, but the reward model needs one to score the next item. Three
substitutes are compared: the expected ratio $\hat\rho_k$, a sample from $\operatorname{Bernoulli}(\hat\rho_k)$, and $1$.

In [ ]:
by_feedback_mode = {}
for mode in SimulatedFeedback:
    evaluation = evaluate(policies["rl"], replace(evaluation_config, simulated_feedback=mode))
    by_feedback_mode[mode.value] = per_step_table(evaluation, test_examples, log.items)["predicted_played_ratio/train"]

pd.DataFrame(by_feedback_mode).plot(marker="o", figsize=(6, 3.5), title="predicted played ratio by list position (RL policy)");

### Top-K against next-K

NDCG@k of the two ways of producing a list, before and after fine-tuning.

In [ ]:
generation_gap = pd.DataFrame({
    (name, list_name): {k: evaluation.metrics[f"{list_name}/all/ndcg@{k}"] for k in evaluation_config.ks}
    for name, evaluation in {"pretrained": kl_sweep["pretrained"], "rl": evaluate(policies["rl"])}.items()
    for list_name in ("top_k", "next_k")
}).rename_axis("NDCG@k")
generation_gap